In [ ]:
# ============================================================
# CELL 0: MERGE ORIGINAL + AUGMENTED INTO ONE POOLED DATASET
# ============================================================
import os
import shutil
from pathlib import Path

ORIGINAL_DIR = "Medicinal Plant Leaf Health Original Dataset"
AUGMENTED_DIR = "Medicinal Plant Leaf Health Augmented Dataset"
MERGED_DIR = "Medicinal Plant Leaf Health FULL Dataset"

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def merge_datasets(original_dir, augmented_dir, merged_dir):
    original_dir = Path(original_dir)
    augmented_dir = Path(augmented_dir)
    merged_dir = Path(merged_dir)

    class_folders = sorted([f.name for f in original_dir.iterdir() if f.is_dir()])
    print(f"Found {len(class_folders)} classes\n")

    for cls in class_folders:
        out_cls_dir = merged_dir / cls
        out_cls_dir.mkdir(parents=True, exist_ok=True)

        orig_count, aug_count = 0, 0

        orig_cls_dir = original_dir / cls
        if orig_cls_dir.exists():
            for f in orig_cls_dir.iterdir():
                if f.is_file() and f.suffix.lower() in VALID_EXTS:
                    shutil.copy2(f, out_cls_dir / f"orig_{f.name}")
                    orig_count += 1

        aug_cls_dir = augmented_dir / cls
        if aug_cls_dir.exists():
            for f in aug_cls_dir.iterdir():
                if f.is_file() and f.suffix.lower() in VALID_EXTS:
                    shutil.copy2(f, out_cls_dir / f"aug_{f.name}")
                    aug_count += 1

        print(f"{cls}: original={orig_count} augmented={aug_count} total={orig_count + aug_count}")

if not os.path.exists(MERGED_DIR):
    merge_datasets(ORIGINAL_DIR, AUGMENTED_DIR, MERGED_DIR)
else:
    print(f"{MERGED_DIR} already exists — skipping merge.")

In [ ]:
# ============================================================
# CELL 1: RANDOM (LEAKAGE-RISK) SPLIT ON THE MERGED DATASET
# ============================================================
import random

random.seed(42)

SOURCE_DIR = MERGED_DIR
SPLIT_OUTPUT_DIR = "Medicinal Plant Leaf Health RANDOM Split Dataset"
SPLITS = {"train": 0.70, "val": 0.15, "test": 0.15}

def random_split_dataset(source_dir, output_dir, splits):
    source_dir = Path(source_dir)
    output_dir = Path(output_dir)

    class_folders = sorted([f for f in source_dir.iterdir() if f.is_dir()])
    print(f"Found {len(class_folders)} classes\n")

    for split_name in splits:
        for cls in class_folders:
            (output_dir / split_name / cls.name).mkdir(parents=True, exist_ok=True)

    grand_total = 0
    for cls in class_folders:
        images = [f for f in cls.iterdir() if f.is_file() and f.suffix.lower() in VALID_EXTS]
        random.shuffle(images)  # ignores original-vs-augmented lineage -- this IS the leakage risk

        n = len(images)
        n_train = int(round(n * splits["train"]))
        n_val = int(round(n * splits["val"]))
        n_test = n - n_train - n_val

        train_imgs = images[:n_train]
        val_imgs = images[n_train:n_train + n_val]
        test_imgs = images[n_train + n_val:]

        for img in train_imgs:
            shutil.copy2(img, output_dir / "train" / cls.name / img.name)
        for img in val_imgs:
            shutil.copy2(img, output_dir / "val" / cls.name / img.name)
        for img in test_imgs:
            shutil.copy2(img, output_dir / "test" / cls.name / img.name)

        grand_total += n
        print(f"{cls.name}: total={n} train={len(train_imgs)} val={len(val_imgs)} test={len(test_imgs)}")

    print(f"\nGrand total images processed: {grand_total}")

if not os.path.exists(SPLIT_OUTPUT_DIR):
    random_split_dataset(SOURCE_DIR, SPLIT_OUTPUT_DIR, SPLITS)
else:
    print(f"{SPLIT_OUTPUT_DIR} already exists — skipping split.")

In [ ]:
# ============================================================
# CELL 2: CONFIG
# ============================================================
import time, json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix, classification_report
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())

DATA_DIR = SPLIT_OUTPUT_DIR   # the RANDOM (leakage-risk) split
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 30
LR = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

RUN_TAG = "custom_cnn_full_RANDOMSPLIT_ablation"
SAVE_DIR = os.path.join("checkpoints", RUN_TAG)
os.makedirs(SAVE_DIR, exist_ok=True)
CKPT_PATH = os.path.join(SAVE_DIR, "last_checkpoint.pth")
BEST_PATH = os.path.join(SAVE_DIR, "best_model.pth")
RESULTS_PATH = os.path.join(SAVE_DIR, "results.json")
CURVES_PATH = os.path.join(SAVE_DIR, "curves.png")
HISTORY_PATH = os.path.join(SAVE_DIR, "history.json")

In [ ]:
# ============================================================
# CELL 3: TRANSFORMS
# ============================================================
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# ============================================================
# CELL 4: LOAD DATA (from the random-leakage split)
# ============================================================
train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_tf)
val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=eval_tf)
test_ds  = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

NUM_CLASSES = len(train_ds.classes)
class_names = train_ds.classes
print(f"Classes ({NUM_CLASSES}): {class_names}")
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
assert train_ds.classes == val_ds.classes == test_ds.classes

In [ ]:
# ============================================================
# CELL 5: CLASS WEIGHTS
# ============================================================
class_counts = np.bincount([label for _, label in train_ds.samples])
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

In [ ]:
# ============================================================
# CELL 6: CUSTOM CNN (same architecture as main run, for fair comparison)
# ============================================================
class LightCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, num_classes))

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

model = LightCNN(NUM_CLASSES).to(DEVICE)
num_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {num_params:,}")

In [ ]:
# ============================================================
# CELL 7: LOSS / OPTIMIZER
# ============================================================
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
# ============================================================
# CELL 8: EPOCH RUNNER
# ============================================================
def run_epoch(loader, train=True, desc="Epoch"):
    model.train() if train else model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    context = torch.enable_grad() if train else torch.no_grad()
    pbar = tqdm(loader, desc=desc, leave=False)
    with context:
        for imgs, labels in pbar:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, macro_f1, precision, recall

In [ ]:
# ============================================================
# CELL 9: CHECKPOINT RESUME
# ============================================================
start_epoch = 1
best_val_f1 = 0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "train_f1": [], "val_f1": []}

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    start_epoch = ckpt["epoch"] + 1
    best_val_f1 = ckpt["best_val_f1"]
    if os.path.exists(HISTORY_PATH):
        with open(HISTORY_PATH) as f:
            history = json.load(f)
    print(f"Resumed at epoch {start_epoch}, best_val_f1={best_val_f1:.4f}")

In [ ]:
# ============================================================
# CELL 10: TRAINING LOOP
# ============================================================
for epoch in range(start_epoch, EPOCHS + 1):
    train_loss, train_acc, train_f1, _, _ = run_epoch(train_loader, train=True, desc=f"Epoch {epoch}/{EPOCHS} [train]")
    val_loss, val_acc, val_f1, val_prec, val_rec = run_epoch(val_loader, train=False, desc=f"Epoch {epoch}/{EPOCHS} [val]")
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)

    print(f"[{RUN_TAG}] Epoch {epoch}/{EPOCHS} | Train acc {train_acc:.4f} f1 {train_f1:.4f} | "
          f"Val acc {val_acc:.4f} f1 {val_f1:.4f} | LR {optimizer.param_groups[0]['lr']:.6f}", flush=True)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), BEST_PATH)
        print(f"  -> New best model saved (val macro-F1={val_f1:.4f})", flush=True)

    torch.save({
        "epoch": epoch, "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
        "best_val_f1": best_val_f1,
    }, CKPT_PATH)

    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=2)

In [ ]:
# ============================================================
# CELL 11: PLOT CURVES
# ============================================================
epochs_range = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(epochs_range, history["train_loss"], label="Train Loss")
axes[0].plot(epochs_range, history["val_loss"], label="Val Loss")
axes[0].set_title(f"{RUN_TAG} - Loss"); axes[0].legend()
axes[1].plot(epochs_range, history["train_acc"], label="Train Acc")
axes[1].plot(epochs_range, history["val_acc"], label="Val Acc")
axes[1].set_title(f"{RUN_TAG} - Accuracy"); axes[1].legend()
axes[2].plot(epochs_range, history["train_f1"], label="Train F1")
axes[2].plot(epochs_range, history["val_f1"], label="Val F1")
axes[2].set_title(f"{RUN_TAG} - Macro-F1"); axes[2].legend()
plt.tight_layout()
plt.savefig(CURVES_PATH, dpi=300)
plt.show()

In [ ]:
# ============================================================
# CELL 12: FINAL TEST EVAL — compare against your leakage-aware custom_cnn_full run
# ============================================================
model.load_state_dict(torch.load(BEST_PATH))
model.eval()

sample_batch, _ = next(iter(test_loader))
sample_batch = sample_batch.to(DEVICE)
with torch.no_grad():
    start = time.time()
    for _ in range(20):
        _ = model(sample_batch)
    elapsed = time.time() - start
per_image_ms = (elapsed / (20 * sample_batch.size(0))) * 1000

test_loss, test_acc, test_f1, test_prec, test_rec = run_epoch(test_loader, train=False, desc="Final Test")
num_params = sum(p.numel() for p in model.parameters())
model_size_mb = num_params * 4 / (1024 ** 2)

results = {
    "model": "custom_cnn", "train_mode": "full_RANDOMSPLIT_ablation",
    "test_accuracy": test_acc, "test_macro_f1": test_f1,
    "test_precision": test_prec, "test_recall": test_rec,
    "total_params": num_params, "trainable_params": num_params,
    "model_size_mb": round(model_size_mb, 3),
    "inference_ms_per_image": round(per_image_ms, 3),
    "best_val_macro_f1": best_val_f1,
}
print("\n=== FINAL TEST RESULTS (RANDOM SPLIT - LEAKAGE RISK) ===")
print(json.dumps(results, indent=2))
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nCompare this against your 'custom_cnn_full' (leakage-aware, original-image-first split) run "
      f"in the comparison table — expect this RANDOM SPLIT version to show INFLATED accuracy due to leakage.")